In [3]:

import asyncio
import time
import re


# ==================================================
# 1. CLASSE LOGABLE
# ==================================================

class Logable:
    def __init__(self, name, verbose=True):
        self.name = name
        self.verbose = verbose

    def log(self, msg):
        if self.verbose:
            print(f"[{self.name}] {msg}", flush=True)


# ==================================================
# 2. CLASSE ACCESSOIRE
# ==================================================

class Accessoire(Logable):
    def __init__(self, name, verbose=True):
        super().__init__(name, verbose)
        self.queue = asyncio.Queue()


# ==================================================
# 3. CLASSE PIC
# ==================================================

class Pic(Accessoire):
    async def embrocher(self, postit):
        await self.queue.put(postit)
        self.log(f"je dépose le post-it '{postit}'")

    async def liberer(self):
        postit = await self.queue.get()
        self.log(f"je récupère le post-it '{postit}'")
        return postit


# ==================================================
# 4. CLASSE BAR
# ==================================================

class Bar(Accessoire):
    async def recevoir(self, commande):
        await self.queue.put(commande)
        self.log(f"je dépose la commande '{commande}'")

    async def evacuer(self):
        commande = await self.queue.get()
        self.log(f"je récupère la commande '{commande}'")
        return commande


# ==================================================
# 5. CLASSE CLIENTS
# ==================================================

class Clients:
    def __init__(self, fname):
        commandes = []
        start = time.time()
        fmt = re.compile(r"(\d+)\s+(.*)")

        with open(fname, "r", encoding="utf-8") as f:
            for line in f:
                found = fmt.search(line)
                if found:
                    when = int(found.group(1))
                    what = found.group(2)
                    commandes.append((start + when, what.split(",")))

        self.commandes = commandes[::-1]

    async def run(self, serveurs):
        i = 0

        while self.commandes:
            when, commande = self.commandes[-1]
            attente = when - time.time()

            if attente > 0:
                await asyncio.sleep(attente)

            self.commandes.pop()

            # Distribution alternée des commandes
            serveur = serveurs[i % len(serveurs)]
            await serveur.prendre_commande(commande)

            i += 1
            await asyncio.sleep(0)

        # Signal de fin pour Bob
        await serveurs[0].pic.embrocher(None)


# ==================================================
# 6. CLASSE EMPLOYE
# ==================================================

class Employe(Logable):
    def __init__(self, pic, bar, clients, name, verbose=True):
        super().__init__(name, verbose)
        self.pic = pic
        self.bar = bar
        self.clients = clients


# ==================================================
# 7. CLASSE BARISTE
# ==================================================

class Bariste(Employe):
    async def preparer(self, commande):
        self.log(f"je commence la préparation de '{commande}'")

        for cocktail in commande:
            self.log(f"je prépare '{cocktail}'")
            await asyncio.sleep(1)

        self.log(f"la commande '{commande}' est prête")
        return commande

    async def run(self):
        while True:
            element = await self.pic.liberer()

            if element is None:
                for serveur in self.serveurs:
                    await serveur.bar.recevoir(None)
                break

            serveur, commande = element

            commande_prete = await self.preparer(commande)
            await serveur.bar.recevoir(commande_prete)


# ==================================================
# 8. CLASSE SERVEUR
# ==================================================

class Serveur(Employe):
    def __init__(self, pic, bar, clients, name, verbose=True):
        super().__init__(pic, bar, clients, name, verbose)
        self.lock = asyncio.Lock()

    async def prendre_commande(self, commande):
        await self.lock.acquire()

        self.log("prêt pour prendre une nouvelle commande...")
        self.log(f"j'ai la commande '{commande}'")
        self.log(f"j'écris sur le post-it '{commande}'")

        await self.pic.embrocher((self, commande))

    async def servir(self, commande):
        self.log(f"j'apporte la commande '{commande}'")

        for cocktail in commande:
            self.log(f"je sers '{cocktail}'")
            await asyncio.sleep(1)

        self.log(f"la commande '{commande}' est servie")
        self.lock.release()

    async def run(self):
        while True:
            commande = await self.bar.evacuer()

            if commande is None:
                break

            await self.servir(commande)


# ==================================================
# 9. FONCTION MAIN
# ==================================================

async def main():
    les_clients = Clients("lundi")

    le_pic = Pic("pic", verbose=True)

    bar_alice = Bar("bar_alice", verbose=True)
    bar_charlie = Bar("bar_charlie", verbose=True)

    alice = Serveur(
        le_pic, bar_alice, les_clients,
        name="alice", verbose=True
    )

    charlie = Serveur(
        le_pic, bar_charlie, les_clients,
        name="charlie", verbose=True
    )

    serveurs = [alice, charlie]

    bob = Bariste(
        le_pic, bar_alice, les_clients,
        name="bob", verbose=True
    )

    bob.serveurs = serveurs

    await asyncio.gather(
        les_clients.run(serveurs),
        bob.run(),
        alice.run(),
        charlie.run()
    )


# ==================================================
# 10. EXECUTION
# ==================================================

await main()


[alice] prêt pour prendre une nouvelle commande...
[alice] j'ai la commande '['planteur', 'piña colada', 'mojito']'
[alice] j'écris sur le post-it '['planteur', 'piña colada', 'mojito']'
[pic] je dépose le post-it '(<__main__.Serveur object at 0x0000017815547CB0>, ['planteur', 'piña colada', 'mojito'])'
[pic] je récupère le post-it '(<__main__.Serveur object at 0x0000017815547CB0>, ['planteur', 'piña colada', 'mojito'])'
[bob] je commence la préparation de '['planteur', 'piña colada', 'mojito']'
[bob] je prépare 'planteur'
[charlie] prêt pour prendre une nouvelle commande...
[charlie] j'ai la commande '['bloody mary', 'moscow mule']'
[charlie] j'écris sur le post-it '['bloody mary', 'moscow mule']'
[pic] je dépose le post-it '(<__main__.Serveur object at 0x00000178142ED810>, ['bloody mary', 'moscow mule'])'
[pic] je dépose le post-it 'None'
[bob] je prépare 'piña colada'
[bob] je prépare 'mojito'
[bob] la commande '['planteur', 'piña colada', 'mojito']' est prête
[bar_alice] je dépose 